# COPIL Presentation Generator
Generateur de presentation PowerPoint depuis Google Drive


In [ ]:
!pip install -q python-pptx
from google.colab import drive
drive.mount('/content/drive')
print('OK - Google Drive monte')

In [ ]:
import os
import csv
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from pptx.enum.chart import XL_CHART_TYPE

DRIVE_FOLDER = '/content/drive/MyDrive/PF_SCORING'

if os.path.exists(DRIVE_FOLDER):
    files = [f for f in os.listdir(DRIVE_FOLDER) if f.lower().endswith('.csv')]
    if files:
        CSV_FILE = os.path.join(DRIVE_FOLDER, files[0])
        print(f'CSV trouve: {files[0]}')
    else:
        print('Aucun CSV dans le dossier')
else:
    print(f'Dossier non trouve: {DRIVE_FOLDER}')

In [ ]:
PRIMARY_BLUE = RGBColor(0, 51, 102)
ACCENT_ORANGE = RGBColor(255, 153, 0)
WHITE = RGBColor(255, 255, 255)
GREEN = RGBColor(76, 175, 80)
ORANGE = RGBColor(255, 152, 0)
RED = RGBColor(244, 67, 54)
DARK_TEXT = RGBColor(33, 33, 33)
LIGHT_GRAY = RGBColor(240, 240, 240)

def parse_completion(item):
    try:
        comp_str = item.get('COMPLÉTION_%', '0').rstrip('%').strip()
        return float(comp_str) if comp_str else 0
    except:
        return 0

def load_csv(csv_file):
    items = []
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        items = list(reader)
    print(f'Charges: {len(items)} elements')
    return items

def calc_stats(items):
    completions = [parse_completion(item) for item in items]
    stats = {
        'total': len(items),
        'integrated': len([i for i in items if i.get('STATUT') == 'INTÉGRÉ']),
        'in_progress': len([i for i in items if i.get('STATUT') == 'EN COURS']),
        'planned': len([i for i in items if i.get('STATUT') == 'PLANIFIÉ']),
        'blocked': len([i for i in items if i.get('STATUT') == 'BLOQUÉ']),
        'by_bloc': {},
        'risks': [i for i in items if i.get('STATUT') == 'BLOQUÉ' or parse_completion(i) < 50]
    }
    stats['average_completion'] = sum(completions) / len(completions) if completions else 0
    for item in items:
        bloc = item.get('BLOC', 'Unknown')
        if bloc not in stats['by_bloc']:
            stats['by_bloc'][bloc] = {'total': 0, 'completion_sum': 0}
        stats['by_bloc'][bloc]['total'] += 1
        stats['by_bloc'][bloc]['completion_sum'] += parse_completion(item)
    for bloc in stats['by_bloc']:
        items_count = stats['by_bloc'][bloc]['total']
        stats['by_bloc'][bloc]['completion'] = stats['by_bloc'][bloc]['completion_sum'] / items_count if items_count > 0 else 0
    return stats

def create_ppt(stats, output_file):
    prs = Presentation()
    prs.slide_width = Inches(10)
    prs.slide_height = Inches(7.5)

    # Slide 1: Title
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.background.fill
    bg.solid()
    bg.fore_color.rgb = PRIMARY_BLUE
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(2.5), Inches(9), Inches(1.5))
    p = title_box.text_frame.paragraphs[0]
    p.text = 'PF SCORING'
    p.font.size = Pt(66)
    p.font.bold = True
    p.font.color.rgb = WHITE
    p.alignment = PP_ALIGN.CENTER

    subtitle_box = slide.shapes.add_textbox(Inches(0.5), Inches(4.2), Inches(9), Inches(1))
    p = subtitle_box.text_frame.paragraphs[0]
    p.text = 'COMITÉ DE PILOTAGE - COPIL'
    p.font.size = Pt(32)
    p.font.color.rgb = ACCENT_ORANGE
    p.alignment = PP_ALIGN.CENTER

    # Slide 2: KPIs
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.background.fill
    bg.solid()
    bg.fore_color.rgb = WHITE

    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(9), Inches(0.5))
    p = title.text_frame.paragraphs[0]
    p.text = 'RÉSUMÉ EXÉCUTIF - KPIs'
    p.font.size = Pt(40)
    p.font.bold = True
    p.font.color.rgb = PRIMARY_BLUE

    kpis = [('INTÉGRÉ', stats['integrated'], GREEN), ('EN COURS', stats['in_progress'], ORANGE), ('PLANIFIÉ', stats['planned'], RGBColor(33, 150, 243)), ('BLOQUÉ', stats['blocked'], RED)]
    for idx, (label, count, color) in enumerate(kpis):
        x = 0.8 + (idx * 2.1)
        shape = slide.shapes.add_shape(1, Inches(x), Inches(1.3), Inches(1.8), Inches(1.5))
        shape.fill.solid()
        shape.fill.fore_color.rgb = color
        shape.line.color.rgb = color

        count_box = slide.shapes.add_textbox(Inches(x), Inches(1.5), Inches(1.8), Inches(0.7))
        p = count_box.text_frame.paragraphs[0]
        p.text = str(count)
        p.font.size = Pt(48)
        p.font.bold = True
        p.font.color.rgb = WHITE
        p.alignment = PP_ALIGN.CENTER

        label_box = slide.shapes.add_textbox(Inches(x), Inches(2.2), Inches(1.8), Inches(0.4))
        p = label_box.text_frame.paragraphs[0]
        p.text = label
        p.font.size = Pt(14)
        p.font.color.rgb = WHITE
        p.alignment = PP_ALIGN.CENTER

    # Slide 3: Completion Chart
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.background.fill
    bg.solid()
    bg.fore_color.rgb = WHITE

    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(9), Inches(0.5))
    p = title.text_frame.paragraphs[0]
    p.text = 'AVANCEMENT PAR BLOC'
    p.font.size = Pt(40)
    p.font.bold = True
    p.font.color.rgb = PRIMARY_BLUE

    chart = slide.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(0.5), Inches(1.1), Inches(9), Inches(5.2)).chart
    chart.has_legend = False
    categories = list(stats['by_bloc'].keys())
    completions = tuple([stats['by_bloc'][bloc]['completion'] for bloc in categories])
    chart.chart_data.categories = categories
    chart.chart_data.add_series('Complétude %', completions)
    series = chart.plots[0].series[0]
    for idx, completion in enumerate(completions):
        color = GREEN if completion >= 90 else ORANGE if completion >= 70 else RED
        series.points[idx].format.fill.solid()
        series.points[idx].format.fill.fore_color.rgb = color

    # Slide 4: Gauge
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.background.fill
    bg.solid()
    bg.fore_color.rgb = WHITE

    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(9), Inches(0.5))
    p = title.text_frame.paragraphs[0]
    p.text = 'COMPLÉTUDE GLOBALE'
    p.font.size = Pt(40)
    p.font.bold = True
    p.font.color.rgb = PRIMARY_BLUE

    completion = stats['average_completion']
    pct_box = slide.shapes.add_textbox(Inches(3), Inches(2), Inches(4), Inches(2))
    p = pct_box.text_frame.paragraphs[0]
    p.text = f'{completion:.0f}%'
    p.font.size = Pt(120)
    p.font.bold = True
    p.font.color.rgb = GREEN if completion >= 80 else ORANGE if completion >= 50 else RED
    p.alignment = PP_ALIGN.CENTER

    # Slide 5: Risks
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.background.fill
    bg.solid()
    bg.fore_color.rgb = WHITE

    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(9), Inches(0.5))
    p = title.text_frame.paragraphs[0]
    p.text = 'RISQUES & BLOCAGES'
    p.font.size = Pt(40)
    p.font.bold = True
    p.font.color.rgb = PRIMARY_BLUE

    risks = stats['risks'][:5]
    if not risks:
        no_risks = slide.shapes.add_textbox(Inches(0.5), Inches(2), Inches(9), Inches(4))
        p = no_risks.text_frame.paragraphs[0]
        p.text = 'Aucun risque identifié'
        p.font.size = Pt(28)
        p.font.color.rgb = GREEN
        p.alignment = PP_ALIGN.CENTER
    else:
        y = 1.3
        for risk in risks:
            box = slide.shapes.add_textbox(Inches(0.7), Inches(y), Inches(8.6), Inches(0.8))
            p = box.text_frame.paragraphs[0]
            element = risk.get('ÉLÉMENT', 'Unknown')
            status = risk.get('STATUT', '')
            comp = parse_completion(risk)
            p.text = f'{element} ({status} - {comp:.0f}%)'
            p.font.size = Pt(14)
            p.font.color.rgb = RED
            y += 0.9

    # Slide 6: Next Steps
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.background.fill
    bg.solid()
    bg.fore_color.rgb = WHITE

    title = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(9), Inches(0.5))
    p = title.text_frame.paragraphs[0]
    p.text = 'PROCHAINES ÉTAPES'
    p.font.size = Pt(40)
    p.font.bold = True
    p.font.color.rgb = PRIMARY_BLUE

    phases = [('Phase 9', 'Design Responsive Mobile'), ('Phase 10', 'Tests & QA Complète'), ('Phase 11', 'Déploiement Production'), ('Phase 12', 'Monitoring & Support')]
    y = 1.3
    for phase, description in phases:
        shape = slide.shapes.add_shape(1, Inches(0.7), Inches(y), Inches(0.6), Inches(0.6))
        shape.fill.solid()
        shape.fill.fore_color.rgb = ACCENT_ORANGE

        box = slide.shapes.add_textbox(Inches(1.5), Inches(y), Inches(7.8), Inches(0.6))
        p = box.text_frame.paragraphs[0]
        p.text = f'{phase} - {description}'
        p.font.size = Pt(16)
        p.font.color.rgb = DARK_TEXT

        y += 1

    prs.save(output_file)
    print(f'PPT sauvegardé: {os.path.basename(output_file)} - 6 slides')

print('Fonctions prêtes')

In [ ]:
if os.path.exists(CSV_FILE):
    items = load_csv(CSV_FILE)
    stats = calc_stats(items)
    output_file = os.path.join(DRIVE_FOLDER, 'COPIL_Presentation.pptx')
    create_ppt(stats, output_file)
    print('Succes! Verifiez votre Google Drive')
else:
    print('Erreur: CSV non trouve')